# Seer — cloud training run

Trains all three Seer models on a free GPU (Colab T4 / Kaggle) and hands back
a small zip of deployable ONNX weights for the CPU serving machine.

**Runtime → Change runtime type → T4 GPU**, then **Run all** — always from the
top. Running a stage cell on its own will fail: the setup cells define the
backup path, clone the repo and install the package.

**Resilient to disconnects.** Every stage checks Google Drive for a prior
result before doing work, and backs up its own result as soon as it finishes.
If your session drops (free Colab reclaims idle/long-running VMs, GPU usage
limits, laptop sleeps), reopen this notebook, connect a GPU runtime, and
**Run all** — finished stages are skipped and only the stage that was in
flight needs redoing.

Everything trained here is synthetic specimen data generated in this
session — nothing sensitive is uploaded or downloaded.

In [ ]:
!nvidia-smi -L

## Mount Drive (the resume point)

Everything that survives a disconnect lives under
`/content/drive/MyDrive/seer_backup/`. Authorize access when prompted.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
BACKUP = "/content/drive/MyDrive/seer_backup"
os.makedirs(BACKUP, exist_ok=True)
print("backup dir:", BACKUP)

In [ ]:
import os, subprocess

if not os.path.isdir("/content/seer"):
    subprocess.run("git clone https://github.com/Lucas-Maingi/seer.git /content/seer",
                   shell=True, check=True)
os.chdir("/content/seer")
subprocess.run("pip install -q -e .", shell=True, check=True)
for d in ("data", "weights", "runs"):
    os.makedirs(d, exist_ok=True)


def stage(cmd: str, produces: str, backup_as: str | None = None) -> None:
    """Run a stage with LIVE output, then verify it actually produced its
    output file before reporting success.

    Uses IPython's system() (the same thing `!cmd` does) so training logs
    stream into the cell as they happen — a long stage with no visible
    output is indistinguishable from a hang. The existence check afterwards
    means a crashed stage still fails loudly instead of printing success.
    """
    assert os.getcwd() == "/content/seer", "run the setup cells first (Run all from the top)"
    if os.path.exists(produces):
        print(f"{produces} already present, skipping")
        return
    get_ipython().system(cmd)   # streams stdout/stderr live into the cell
    if not os.path.exists(produces):
        raise RuntimeError(
            f"FAILED: {cmd}\n"
            f"expected output {produces!r} was not created — see the error above. "
            "Fix it and re-run this cell; nothing was backed up.")
    if backup_as:
        subprocess.run(f'cp -r "{produces}" "{BACKUP}/{backup_as}"', shell=True, check=True)
        print(f"OK: {produces} -> Drive backup ({backup_as})")
    else:
        print(f"OK: {produces}")


print("setup complete, cwd:", os.getcwd())

In [ ]:
# Restore whatever a previous (interrupted) run already finished.
import os, shutil

def restore(name, dest):
    src = f"{BACKUP}/{name}"
    if os.path.exists(src) and not os.path.exists(dest):
        print(f"restoring {name} from Drive ...")
        (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dest)

restore("synth", "data/synth")
for f in ("corner.pt", "crnn.pt", "tamper.pt"):
    restore(f, f"weights/{f}")

print("dataset present:", os.path.exists("data/synth/index.jsonl"))
print("checkpoints present:",
      [f for f in ("corner.pt", "crnn.pt", "tamper.pt") if os.path.exists(f"weights/{f}")])

## Epoch budget

Defaults are tuned to observed convergence on this synthetic data, not to
round numbers. From a full run's logs: OCR reached CER 0.0031 / 98.7% exact
match by **epoch 4** and was flat by epoch 9 — the remaining 30+ epochs cost
half an hour of GPU time to move the fourth decimal place. Localization is
similar (2.4px by epoch 19, 2.26px at 30).

Raise these only if you have session time to burn.

In [ ]:
EPOCHS_CORNER = int(os.environ.get("EPOCHS_CORNER", 20))
EPOCHS_CRNN   = int(os.environ.get("EPOCHS_CRNN", 12))
EPOCHS_TAMPER = int(os.environ.get("EPOCHS_TAMPER", 15))
print(f"corner {EPOCHS_CORNER}, ocr {EPOCHS_CRNN}, tamper {EPOCHS_TAMPER}")

## 1/6 — Generate the synthetic dataset

~8k samples is a good balance for models this small (raise `COUNT` if you have
session time to spare — generation is CPU-bound, ~1.4 s/sample, ~1h45m at 8k).
Skipped automatically if a dataset was restored from Drive.

Optional realism upgrade for the portraits: upload a folder of synthetic faces
(e.g. a slice of the [SFHQ dataset](https://github.com/SelfishGene/SFHQ-dataset))
and set `FACES` below.

In [ ]:
COUNT = int(os.environ.get("COUNT", 8000))
FACES = os.environ.get("FACES", "")   # e.g. "/content/faces"
faces_flag = f"--faces {FACES}" if FACES else ""

stage(f"python -m seer.synth.generate --out data/synth --count {COUNT} {faces_flag}",
      produces="data/synth/index.jsonl")

# back up the whole dataset dir (the expensive artifact) once it exists
if not os.path.exists(f"{BACKUP}/synth/index.jsonl"):
    print("copying dataset to Drive (this takes a few minutes) ...")
    subprocess.run(f'cp -r data/synth "{BACKUP}/synth"', shell=True, check=True)
    print("dataset backed up to Drive")

## 2/6 — Corner localization

In [ ]:
stage(f"python -m seer.localize.train --data data/synth "
      f"--epochs {EPOCHS_CORNER} --out weights/corner.pt",
      produces="weights/corner.pt", backup_as="corner.pt")

## 3/6 — Field OCR

In [ ]:
stage(f"python -m seer.ocr.train --data data/synth "
      f"--epochs {EPOCHS_CRNN} --out weights/crnn.pt",
      produces="weights/crnn.pt", backup_as="crnn.pt")

## 4/6 — Tamper forensics

In [ ]:
stage(f"python -m seer.forensics.train --data data/synth "
      f"--epochs {EPOCHS_TAMPER} --out weights/tamper.pt",
      produces="weights/tamper.pt", backup_as="tamper.pt")

## 5/6 — ONNX export + INT8 quantization

Cheap (seconds to a couple of minutes) — always safe to re-run.

In [ ]:
!python -m seer.runtime.export --all
!python -m seer.runtime.quantize --all --calib data/synth
subprocess.run(f'cp weights/*.onnx "{BACKUP}/"', shell=True)
print("exported:", sorted(f for f in os.listdir("weights") if f.endswith(".onnx")))

## 6/6 — Latency benchmark (indicative only — re-run on the serving CPU)

In [ ]:
!python -m seer.runtime.bench --report runs/latency.md
subprocess.run(f'cp runs/latency.md "{BACKUP}/latency.md"', shell=True)

## Package the deployable weights

Only the ONNX files (and training checkpoints if you want to resume later)
need to leave this machine — a few tens of MB.

In [ ]:
!zip -j seer_weights.zip weights/*.onnx weights/*.pt runs/latency.md
from google.colab import files
files.download("seer_weights.zip")

## Back on the laptop

```bash
unzip seer_weights.zip -d weights/
python scripts/fetch_models.py          # YuNet + ArcFace for the face stage
python -m seer.runtime.bench            # re-bench on the real serving CPU
uvicorn seer.api.main:app --port 8000   # and in web/: npm run dev
```

The latency report that matters is the one measured on the serving hardware,
not this VM — rerun `seer.runtime.bench` locally and commit that.